# Variant Scoring with the Nucleotide Transformer on a T4 GPU

**Workshop: AI on AnVIL — genomic language models for variant effect prediction**

This notebook demonstrates *zero-shot* variant effect scoring with a pretrained DNA
language model (the Nucleotide Transformer, InstaDeepAI). The idea:

- The model is trained only to predict masked nucleotides in genomic sequence — it has
  never seen a disease label.
- If we mask the exact position of a variant and ask the model "how likely is the
  reference base here vs. the alternate base?", a large gap (reference strongly
  preferred over alternate) suggests the alternate allele disrupts a sequence pattern
  the model learned to be important — a proxy for a functional/deleterious effect.
- No variant-specific training is required, which is what makes this a compelling
  illustration of an LLM-style genomic model versus a conventional supervised classifier.

**Steps:**
1. Check GPU availability (T4) and load a mid-size Nucleotide Transformer checkpoint.
2. Fetch a handful of real, well-known human variants (with reference sequence context
   and ClinVar-style significance) from the Ensembl REST API.
3. Score each variant with the model's masked-token log-likelihood ratio.
4. Compare scores against each variant's reported clinical significance.
5. See how to point this at a VCF from your own AnVIL workspace instead of the toy list.

> **Caveat for the workshop:** this is an illustrative, tiny-N demo, not a validated
> clinical classifier. Zero-shot genomic LM scores are a research tool, not a
> replacement for CADD, AlphaMissense, SpliceAI, REVEL, or ACMG-based clinical review.

## 1. Check the GPU

Confirm the runtime has a GPU (a T4 is plenty for the 500M-parameter model used below).
If this doesn't show a GPU, attach one to the notebook's runtime before continuing —
everything below will still run on CPU, just much more slowly.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

### A note on old drivers (e.g. NVIDIA 535)

Driver 535.x tops out at CUDA runtime 12.2 (check the "CUDA Version" field in the
`nvidia-smi` output above). The default `pip install torch` grabs the newest PyTorch
release, which is typically compiled against a newer CUDA runtime (12.4+) and requires
driver 550+ — you'll see `CUDA initialization: The NVIDIA driver on your system is too
old` at runtime even though `torch.cuda.is_available()` may still return `True`.

This is the same class of fix as pinning the Ollama release for stale drivers: install a
PyTorch build compiled against a CUDA runtime the driver actually supports, using
PyTorch's versioned wheel index instead of the default PyPI index. `cu121` (CUDA 12.1)
is safely within what a 535.x driver supports.

In [ ]:
# Pin torch to a build compiled against CUDA 12.1, which driver 535.x can run
# (its max supported CUDA runtime is 12.2). Skip this and use a plain
# `pip install torch` if your driver is 550+ / CUDA 12.4+.
%pip install -q --index-url https://download.pytorch.org/whl/cu121 torch==2.5.1

# Pin transformers below 5.0: this model's remote code (modeling_esm.py, pulled
# live from the Hugging Face Hub via trust_remote_code) still imports a helper
# function from transformers.pytorch_utils that was removed in the 5.x line,
# which raises "ImportError: cannot import name 'find_pruneable_heads_and_indices'"
# on a freshly-installed, latest transformers. InstaDeepAI's own repo pins
# transformers>=4.52.4, so this stays on the newest working 4.x release.
%pip install -q "transformers>=4.52,<5" accelerate requests matplotlib

In [ ]:
import torch

print(f"torch {torch.__version__} (built for CUDA {torch.version.cuda})")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA not available. If nvidia-smi shows a GPU but this still says 'cpu', "
          "the installed torch build's CUDA version is likely newer than the driver "
          "supports - re-check the pinned cu121 install above against your driver's "
          "max CUDA version.")

## 2. Load the Nucleotide Transformer

Using the 500M-parameter multi-species v2 checkpoint — a real, capable genomic LM that's
still small enough to load comfortably on a T4.

`trust_remote_code` is required because InstaDeepAI ships a custom tokenizer/model
implementation. That custom code (an ESM-style architecture with rotary position
embeddings) builds its sin/cos position tables as plain float32 tensors regardless of
the model's overall dtype. If the model is loaded with `torch_dtype=torch.float16`,
those float32 position tables get combined with float16 hidden states inside attention
and PyTorch raises `RuntimeError: expected scalar type Half but found Float` — and it
can appear to fail only for *some* variants, since it depends on which rotary-table
code path a given sequence length/position happens to hit.

The simplest fix is to just not force fp16: load in the default float32. A 500M-param
model in float32 is only ~2 GB of weights, trivial for a 16 GB T4, so there's no real
cost to sidestepping the bug for a demo like this.

If this checkpoint name has moved, browse current options at
https://huggingface.co/InstaDeepAI and swap in the model id below (e.g. the
`nucleotide-transformer-500m-human-ref` variant, trained only on the human reference,
is a reasonable alternative for a human-variant-only demo).

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_NAME = "InstaDeepAI/nucleotide-transformer-v2-500m-multi-species"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    # Deliberately NOT torch.float16 - this model's rotary position-embedding
    # buffers are built in float32 internally and mixing them with float16
    # hidden states raises "expected scalar type Half but found Float" for
    # some inputs. Float32 avoids the bug; the model is small enough that the
    # extra memory/compute cost doesn't matter here.
).to(device)
model.eval()

mask_token_id = tokenizer.mask_token_id
n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL_NAME} ({n_params / 1e6:.0f}M params)")
print(f"Mask token: {tokenizer.mask_token!r} (id={mask_token_id})")

## 3. Pull real variants + reference context from Ensembl

Rather than hard-coding genomic coordinates (easy to get subtly wrong), this fetches
each variant live from the Ensembl REST API by rsID: its GRCh38 coordinates, its
ClinVar-derived clinical significance (when Ensembl has one on file), and a window of
flanking reference sequence.

Two wrinkles specific to using rsIDs (rather than a VCF, where ref/alt are already
fully specified) are handled explicitly:

- **Strand.** `allele_string` can be reported relative to either genomic strand, while
  fetched sequence is always plus-strand, so alleles are flipped when needed.
- **Multiallelic rsIDs.** Old, heavily-studied rsIDs (rs334, rs6025, rs1801133, ...)
  often accumulate more than two alleles in dbSNP over the years as different studies
  report additional rare variants at the same position under the same ID - e.g.
  `T/A/C/G` instead of a clean `ref/alt` pair. Rather than skip these, the true
  reference base is read directly from the fetched sequence (unambiguous ground
  truth), and when more than one non-reference allele remains, Ensembl's reported
  `minor_allele` is used to pick the one this rsID is generally taken to refer to. If
  that heuristic can't resolve it, the first remaining candidate is used and flagged
  as a guess - this is exactly the kind of ambiguity that disappears once you're
  scoring variants straight from a VCF's REF/ALT columns instead of an rsID lookup
  (see Section 8).

In [ ]:
import requests

ENSEMBL_REST = "https://rest.ensembl.org"
WINDOW = 100  # bases of context on each side of the variant
COMPLEMENT = str.maketrans("ACGT", "TGCA")


def fetch_variant_context(rsid, window=WINDOW, forced_alt=None):
    """Resolve an rsID to GRCh38 coordinates + flanking reference sequence via Ensembl.

    `forced_alt` lets you override the automatic alt-allele choice for a known
    multiallelic rsID, e.g. fetch_variant_context("rs334", forced_alt="T").

    Returns a dict, or None if the variant can't be resolved as a single-nucleotide
    substitution.
    """
    r = requests.get(
        f"{ENSEMBL_REST}/variation/human/{rsid}",
        params={"content-type": "application/json"},
        timeout=15,
    )
    if r.status_code != 200:
        print(f"[{rsid}] lookup failed: HTTP {r.status_code}")
        return None
    data = r.json()

    mappings = [
        m for m in data.get("mappings", [])
        if str(m.get("assembly_name", "")).startswith("GRCh38")
    ]
    if not mappings:
        print(f"[{rsid}] no GRCh38 mapping found")
        return None
    mapping = mappings[0]
    chrom = mapping["seq_region_name"]
    pos = mapping["start"]
    strand = mapping.get("strand", 1)

    # Keep only single-base alleles - drops any indel/complex entries merged into
    # the same rsID, which we can't score with this simple substitution method.
    alleles = [a for a in mapping["allele_string"].split("/") if len(a) == 1]
    if strand == -1:
        alleles = [a.translate(COMPLEMENT) for a in alleles]
    if len(alleles) < 2:
        print(f"[{rsid}] no usable single-nucleotide alleles in "
              f"({mapping['allele_string']}); skipping")
        return None

    seq_r = requests.get(
        f"{ENSEMBL_REST}/sequence/region/human/{chrom}:{pos - window}-{pos + window}",
        params={"content-type": "text/plain"},
        timeout=15,
    )
    if seq_r.status_code != 200:
        print(f"[{rsid}] sequence fetch failed: HTTP {seq_r.status_code}")
        return None
    flank = seq_r.text.strip().upper()

    center = window  # index of the variant base within `flank`
    ref = flank[center]
    if ref not in alleles:
        print(f"[{rsid}] reference base {ref} not among reported alleles {alleles}; skipping")
        return None

    candidates = [a for a in alleles if a != ref]
    if not candidates:
        print(f"[{rsid}] no alternate allele distinct from reference; skipping")
        return None

    if forced_alt is not None:
        alt = forced_alt.upper()
    elif len(candidates) == 1:
        alt = candidates[0]
    else:
        minor = data.get("minor_allele")
        if minor and strand == -1:
            minor = minor.translate(COMPLEMENT)
        if minor in candidates:
            alt = minor
            print(f"[{rsid}] multiallelic ({alleles}); using minor_allele {alt}")
        else:
            alt = candidates[0]
            print(f"[{rsid}] multiallelic ({alleles}) with no usable minor_allele match; "
                  f"defaulting to {alt} - pass forced_alt= to override")

    return {
        "rsid": rsid,
        "chrom": chrom,
        "pos": pos,
        "ref": ref,
        "alt": alt,
        "clinical_significance": data.get("clinical_significance", []),
        "ref_sequence": flank,
        "alt_sequence": flank[:center] + alt + flank[center + 1:],
    }

## 4. Example variant set

A handful of well-known, real human SNPs — deliberately resolved live via the API
above rather than assumed, so coordinates and significance labels are always current.
Swap these rsIDs for anything relevant to your audience.

In [ ]:
EXAMPLE_RSIDS = [
    "rs334",       # HBB - sickle cell allele
    "rs6025",      # F5 - Factor V Leiden
    "rs7412",      # APOE
    "rs429358",    # APOE
    "rs1800562",   # HFE - hemochromatosis (C282Y)
    "rs1801133",   # MTHFR - C677T
    "rs1799990",   # PRNP - M129V
]

variants = []
for rsid in EXAMPLE_RSIDS:
    v = fetch_variant_context(rsid)
    if v is not None:
        variants.append(v)

print(f"Resolved {len(variants)}/{len(EXAMPLE_RSIDS)} variants")
variants

## 5. Zero-shot scoring

Tokenize the reference and alternate sequences independently, find the single token
that differs between them, mask that position in the reference sequence, and read off
the model's log-probability for the reference token vs. the alternate token at that
masked position. The score is `log P(alt) - log P(ref)`: more negative means the model
finds the alternate allele much less likely than the reference in that context.

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def score_variant(ref_sequence, alt_sequence):
    ref_ids = tokenizer(ref_sequence, return_tensors="pt")["input_ids"][0]
    alt_ids = tokenizer(alt_sequence, return_tensors="pt")["input_ids"][0]

    if ref_ids.shape != alt_ids.shape:
        raise ValueError("ref/alt tokenized to different lengths - unexpected for an SNV")

    diff_positions = (ref_ids != alt_ids).nonzero(as_tuple=True)[0]
    if len(diff_positions) != 1:
        raise ValueError(f"expected exactly one differing token, found {len(diff_positions)}")
    idx = diff_positions[0].item()

    ref_token_id = ref_ids[idx].item()
    alt_token_id = alt_ids[idx].item()

    masked_ids = ref_ids.clone()
    masked_ids[idx] = mask_token_id
    input_ids = masked_ids.unsqueeze(0).to(device)

    logits = model(input_ids=input_ids).logits[0, idx]
    log_probs = F.log_softmax(logits.float(), dim=-1)

    return (log_probs[alt_token_id] - log_probs[ref_token_id]).item()

In [ ]:
results = []
for v in variants:
    try:
        s = score_variant(v["ref_sequence"], v["alt_sequence"])
    except Exception as e:
        print(f"[{v['rsid']}] scoring failed: {e}")
        continue
    sig = ", ".join(v["clinical_significance"]) if v["clinical_significance"] else "(none reported)"
    results.append({**v, "score": s, "clinsig_label": sig})
    print(f"{v['rsid']:<12} {v['chrom']}:{v['pos']:<12} {v['ref']}>{v['alt']}  "
          f"score={s:+.3f}   ClinVar: {sig}")

## 6. Plot: do scores separate pathogenic from benign?

In [ ]:
import matplotlib.pyplot as plt

labels = [r["rsid"] for r in results]
scores = [r["score"] for r in results]
sigs = [r["clinsig_label"] for r in results]
colors = ["#c0392b" if "pathogenic" in s.lower() else "#2980b9" for s in sigs]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, scores, color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("log P(alt) - log P(ref)")
ax.set_title("Zero-shot Nucleotide Transformer variant scores")
plt.xticks(rotation=30, ha="right")
for bar, sig in zip(bars, sigs):
    ax.annotate(sig, (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                textcoords="offset points", xytext=(0, 4), ha="center",
                fontsize=7, rotation=90)
plt.tight_layout()
plt.show()

## 7. Discussion points for the workshop

- **Tiny N, not a benchmark.** Seven variants is enough to show the mechanics, not to
  claim the model "works." A real evaluation would score thousands of ClinVar
  pathogenic/benign SNVs and report AUROC, as the Nucleotide Transformer, GPN, and
  related papers do.
- **`clinical_significance` comes straight from Ensembl's ClinVar mirror** and can be
  sparse, outdated, or conflicting for some IDs — worth a live caveat if a bar shows
  "(none reported)".
- **Strand handling matters.** The mismatch check and strand-flip in
  `fetch_variant_context` are there because skipping this step silently produces wrong
  reference alleles for minus-strand submissions — a good moment to flag common
  correctness pitfalls in genomics pipelines generally.
- **This is a proxy, not a diagnosis.** A low score means "the model finds this allele
  surprising in its sequence context," which correlates with functional importance but
  is not equivalent to clinical pathogenicity.

## 8. Scaling this to your own AnVIL workspace

Swap the toy `EXAMPLE_RSIDS` list for variants pulled directly from a workspace VCF,
and read reference sequence locally instead of one Ensembl call per variant (much
faster for a full cohort):

```python
# from cyvcf2 import VCF
#
# vcf_path = "gs://fc-xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx/path/to/cohort.vcf.gz"
# workspace_variants = []
# for rec in VCF(vcf_path):
#     if len(rec.REF) == 1 and len(rec.ALT[0]) == 1:
#         workspace_variants.append({
#             "chrom": rec.CHROM.replace("chr", ""),
#             "pos": rec.POS,
#             "ref": rec.REF,
#             "alt": rec.ALT[0],
#         })
#
# Then replace the Ensembl sequence lookup with a local reference FASTA read
# (e.g. pyfaidx against the workspace's reference genome) to avoid one HTTP
# request per variant when scoring an entire cohort.
```

**GPU note:** the 500M model comfortably fits a T4 in fp16 with room to batch dozens of
variants at once (stack multiple masked sequences into one forward pass) if you want to
scale this beyond a one-at-a-time loop.